In [1]:
import pandas as pd
import pyterrier as pt
import math
import warnings
import itertools
import pyterrier as pt
from collections import defaultdict
from pyterrier.model import add_ranks
import torch
from torch.nn import functional as F
from transformers import T5Tokenizer, T5ForConditionalGeneration, MT5ForConditionalGeneration
import os
from tqdm import tqdm
import json
import gzip
import pickle

/opt/miniconda3/envs/tmp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_CHOICE = "strategy1"
CUTOFF = 10
OUTPUT_PATH = "./outputs/test_output.json"

queries = pd.read_csv("../data/training_queries.tsv", sep = "\t", names = ["qid", "query"])
text = pd.read_csv("../data/text.tsv", sep = "\t", names = ["docno", "text"])
queries = pt.model.coerce_dataframe_types(queries)
text = pt.model.coerce_dataframe_types(text)

def get_text(run):
    run = pd.merge(run, text, on = "docno")
    return run

In [3]:
bm25_res = pt.io.read_results("./bm25_k1000_training_data.run.gz")

In [4]:
bm25_res = pt.Transformer.from_df(bm25_res)

In [5]:
if MODEL_CHOICE == "strategy1":
    model_path = "/nfs/primary/sas_cross_encoder/trained_models/ohsumed_randomneg_synth_data_monot5_sas_best-1"
elif MODEL_CHOICE == "strategy2":
    model_path = "/nfs/primary/sas_cross_encoder/trained_models/ohsumed_1bm25_proportionate_neg_synth_data_monot5_sas_random_negs_best-31"
elif MODEL_PATH == "strategy3":
    model_path = "/nfs/primary/sas_cross_encoder/trained_models/ohsumed_synth_data_proportionate_negs_sas_easy_negs_best-21"
else:
    raise ValueError(f"Invalid strategy: {MODEL_CHOICE}")

In [6]:
class MonoT5ReRanker(pt.Transformer):
    def __init__(self, 
                 tok_model='t5-base',
                 model='castorini/monot5-base-msmarco',
                 batch_size=4,
                 text_field='text',
                 verbose=True,
                 output_path = None):
        self.verbose = verbose
        self.batch_size = batch_size
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = T5Tokenizer.from_pretrained(tok_model)
        self.model_name = model
        self.model = T5ForConditionalGeneration.from_pretrained(model)
        self.model.to(self.device)
        self.model.eval()
        self.text_field = text_field
        self.REL = self.tokenizer.encode('true')[0]
        self.NREL = self.tokenizer.encode('false')[0]
        self.output_path = output_path

    def __str__(self):
        return f"MonoT5({self.model_name})"

    def transform(self, run):
            scores = []
            queries, texts = run['query'], run[self.text_field]
            it = range(0, len(queries), self.batch_size)
            prompts = self.tokenizer.batch_encode_plus(
                ['Relevant:' for _ in range(self.batch_size)],
                return_tensors='pt', 
                padding='longest'
            )
            max_vlen = self.model.config.n_positions - prompts['input_ids'].shape[1]
    
            # Prepare file for incremental saving
            output_file = self.output_path
                
            with open(output_file, 'a') as f:
                first_batch = os.path.getsize(output_file) == 0  # Check if file is empty (first batch)
                
                if first_batch:
                    f.write('[')  # Start the JSON array
                
                if self.verbose:
                    it = tqdm(it, desc='monoT5', unit='batches')
    
                for start_idx in it:
                    rng = slice(start_idx, start_idx + self.batch_size)
                    enc = self.tokenizer.batch_encode_plus(
                        [f'Query: {q} Document: {d}' for q, d in zip(queries[rng], texts[rng])],
                        return_tensors='pt', 
                        padding='longest'
                    )
                    for key, enc_value in list(enc.items()):
                        enc_value = enc_value[:, :-1]  # Chop off end of sequence token
                        enc_value = enc_value[:, :max_vlen]  # Truncate to max length
                        enc[key] = torch.cat([enc_value, prompts[key][:enc_value.shape[0]]], dim=1)  # Add prompt
    
                    enc['decoder_input_ids'] = torch.full(
                        (len(queries[rng]), 1),
                        self.model.config.decoder_start_token_id,
                        dtype=torch.long
                    )
                    enc = {k: v.to(self.device) for k, v in enc.items()}
    
                    with torch.no_grad():
                        result = self.model(**enc).logits
                    result = result[:, 0, (self.REL, self.NREL)]
                    scores += F.log_softmax(result, dim=1)[:, 0].cpu().detach().tolist()
    
                    current_slice = run.iloc[start_idx:start_idx + self.batch_size]
                    out_qids = current_slice.qid.tolist()
                    out_docnos = current_slice.docno.tolist()
                    out_scores = scores[start_idx:start_idx + self.batch_size]
    
                    # Write the batch results incrementally to the file
                    batch_results = []
                    for qid, docno, score in zip(out_qids, out_docnos, out_scores):
                        batch_results.append({
                            "qid": qid,
                            "docno": docno,
                            "score": score
                        })
                    
                    # If not the first batch, write a comma before appending
                    if not first_batch:
                        f.write(',\n')
    
                    # Write the results for this batch to the file
                    json.dump(batch_results, f)
                    
                    # Set first_batch to False after the first batch is processed
                    first_batch = False
    
                # Close the JSON array after the last batch
                f.write(']')
            
            run = run.drop(columns=['score', 'rank'], errors='ignore').assign(score=scores)
            run = add_ranks(run)
            return run
            
ce = MonoT5ReRanker(model = model_path, output_path = OUTPUT_PATH)
pipeline = bm25_res % CUTOFF >> pt.apply.generic(get_text) >> ce
pipeline(queries.head(2))

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
monoT5: 100%|██████████| 5/5 [00:03<00:00,  1.53batches/s]


,qid,query,docno,name,text,score,rank
0,0,does acetaldehyde react with ribonuclease A,88000001,pyterrier,The binding of acetaldehyde to the active site...,-0.027824,0
1,0,does acetaldehyde react with ribonuclease A,91282060,pyterrier,Binding of anti-acetaldehyde IgG antibodies to...,-7.012740,4
2,0,does acetaldehyde react with ribonuclease A,88192767,pyterrier,Acetaldehyde adducts with haemoglobin: determi...,-8.414424,7
3,0,does acetaldehyde react with ribonuclease A,88106866,pyterrier,Functional consequences of acetaldehyde bindin...,-7.514879,6
4,0,does acetaldehyde react with ribonuclease A,89164696,pyterrier,Chemiluminescence from acetaldehyde oxidation ...,-9.647251,9
5,0,does acetaldehyde react with ribonuclease A,88234504,pyterrier,Nucleoside adducts are formed by cooperative r...,-0.576821,1
6,0,does acetaldehyde react with ribonuclease A,88056756,pyterrier,Antibodies against acetaldehyde-modified prote...,-7.244375,5
7,0,does acetaldehyde react with ribonuclease A,88269057,pyterrier,Acetaldehyde-albumin adduct formation: possibl...,-4.569441,2
8,0,does acetaldehyde react with ribonuclease A,91345655,pyterrier,Drug-acetaldehyde interactions during ethanol ...,-8.475860,8
9,0,does acetaldehyde react with ribonuclease A,91207506,pyterrier,Antibodies against acetaldehyde-modified epito...,-6.653794,3


In [7]:
def transform_data_to_ce_format(data):
    res = {}
    # Iterate over each sublist in the data
    for sublist in data:
        # Iterate over each dictionary in the sublist
        for entry in sublist:
            qid = str(entry['qid'])  # Convert qid to string
            docno = str(entry['docno'])  # Convert docno to string
            score = entry['score']
            
            # Initialize the dictionary for the qid if it doesn't exist
            if qid not in res:
                res[qid] = {}
                
            # Assign the score to the docno for the given qid
            res[qid][docno] = score
    
    return res

def convert():
    with open(OUTPUT_PATH, 'r') as file:
        data = json.load(file)
    
    data = transform_data_to_ce_format(data)
    
    with gzip.open(f'{OUTPUT_PATH}.pkl.gz', 'wb') as f:
        pickle.dump(data, f)

convert()

JSONDecodeError: Extra data: line 5 column 262 (char 1305)

In [18]:
class MyReranker(pt.Transformer):
  def transform_iter(self, inp):
    for query in inp:
        for doc in docs:
            yield doc

def test(run):
    print(run)
    
myreranker = MyReranker()

In [12]:
docs = text.to_dict(orient = "records")
queries = queries.to_dict(orient = "records")

In [23]:
res = myreranker(queries)

KeyboardInterrupt: 